# Exercise 2: Spatially join locations to census areas

This exercise focuses on the logic of a spatial join.

## What we have

**Locations**

| name | geometry |
|---|---|
| Library A | POINT(...) |
| Library B | POINT(...) |

**Census areas**

| ADAUID | population | geometry |
|---|---:|---|
| Area 1 | 4,500 | POLYGON(...) |
| Area 2 | 6,200 | POLYGON(...) |

## What we want

| name | ADAUID | population |
|---|---|---:|
| Library A | Area 1 | 4,500 |
| Library B | Area 2 | 6,200 |

There is no shared ID in the location data. GeoPandas determines which polygon contains each point.

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt

In [ ]:
libraries = gpd.read_file(
    "../data/practice/toronto_library_points_practice.geojson"
)

census = gpd.read_file(
    "../data/raw/toronto_census_areas.geojson"
)

## 1. Inspect the two layers

Print:

- row count;
- CRS;
- geometry types.

Confirm that both datasets are plausible before joining.

In [ ]:
print("Libraries:")
print("Rows:", len(libraries))
print("CRS:", libraries.crs)
print(libraries.geometry.geom_type.value_counts())

print("\nCensus:")
print("Rows:", len(census))
print("CRS:", census.crs)
print(census.geometry.geom_type.value_counts())

## 2. Align the CRS

Reproject `libraries` to `census.crs`.

Store the result as `library_points`.

In [ ]:
library_points = libraries.to_crs(
    census.crs
).copy()

## 3. Perform the spatial join

Attach these census fields:

```text
ADAUID
pop_2021_count
hh_no_veh_pct
```

Use:

```python
how="left"
predicate="within"
```

Store the result as `libraries_with_census`.

In [ ]:
libraries_with_census = gpd.sjoin(
    library_points,
    census[
        [
            "ADAUID",
            "pop_2021_count",
            "hh_no_veh_pct",
            "geometry"
        ]
    ],
    how="left",
    predicate="within"
)

## 4. Check the join

A spatial join should be checked just like a regular merge.

Ask:

- Did the number of library records change?
- How many libraries have no `ADAUID`?
- Do the attached census fields look reasonable?

In [ ]:
print("Rows before join:", len(library_points))
print("Rows after join:", len(libraries_with_census))
print(
    "Missing ADAUID:",
    libraries_with_census["ADAUID"].isna().sum()
)

display(
    libraries_with_census[
        [
            "name",
            "ADAUID",
            "pop_2021_count",
            "hh_no_veh_pct"
        ]
    ].head()
)

## 5. Count libraries by census area

Your small summary should look conceptually like:

| ADAUID | library_count |
|---|---:|
| Area A | 2 |
| Area B | 1 |
| Area C | 3 |

Use `groupby()` and `.size()`.

In [ ]:
library_counts = (
    libraries_with_census
    .groupby("ADAUID")
    .size()
    .reset_index(name="library_count")
)

display(library_counts.head())

## 6. Merge counts back to the census polygons

Use a **left merge** so every census area remains.

After the merge, areas with no matched libraries will have a missing `library_count`.

Once you have checked the join, replace those missing counts with `0`.

In [ ]:
census_with_libraries = census.merge(
    library_counts,
    on="ADAUID",
    how="left"
)

census_with_libraries["library_count"] = (
    census_with_libraries["library_count"]
    .fillna(0)
)

print("Rows after merge:", len(census_with_libraries))
print(
    "Missing library counts:",
    census_with_libraries["library_count"].isna().sum()
)

## 7. Create a simple map

Map `library_count`.

Include a legend, title, and hidden axes.

In [ ]:
ax = census_with_libraries.plot(
    column="library_count",
    legend=True,
    edgecolor="white",
    linewidth=0.2,
    figsize=(8, 8)
)

ax.set_title("Library locations by Toronto census area")
ax.set_axis_off()